In [1]:
from pydoc import doc
from openai import OpenAI
from pathlib import Path
import os
import sys
from dotenv import load_dotenv
from pypdf import PdfReader
import chromadb
import traceback

lib_path = str(Path.cwd().parent.parent)
sys.path.append(lib_path)

from library.common_bot import dump_as_json

basepath = Path.cwd().parent.parent
cwd = basepath.joinpath("env/.env")
bookPath = basepath.joinpath("public/books")
chroma_db_path = basepath.joinpath("chroma_db_path")

if not bookPath.exists():
    sys.exit("Books folder not found")

if cwd.exists():
    load_dotenv(dotenv_path=cwd, override=True)
else:
    print("No .env file found")
    sys.exit(1)

print(os.environ["APP_NAME"])


Python Practice Combined


In [2]:
from openai import OpenAI
googleBaseUrl = os.getenv("GEMINI_BASEURL")
model = os.getenv("GEMINI_EMBEDDING_MODEL_2")
gemini_key = os.getenv("GEMINI_API_KEY")
# print(model)
# print(googleBaseUrl)
client = OpenAI(base_url=googleBaseUrl, api_key=gemini_key)

def embed_text_func(chunks: list[str]) -> list[list[float]] | None:
    """
    Generates embeddings for a list of text chunks.
    Returns a list of embeddings (each embedding is a list of floats).
    Returns an empty list if an error occurs.
    """
    try:
        response = client.embeddings.create(
            input=chunks,
            model=model,
        )
        return [d.embedding for d in response.data]
    except Exception as e:
        print(f"Error generating embeddings: {e}")
        return []

# embed_text_func("embed following text")

In [3]:
"""Local Embeddding"""
ollama_embedding_baseurl = os.getenv("OLLAMA_BASE_URL")
ollama_embedding_model= os.getenv("OLLAMA_EMBEDDING_MODEL")
ollama_client = OpenAI(base_url=ollama_embedding_baseurl, api_key="OLLAMA")

def embed_text_ollama_func(chunks: str) -> list[list[float]] | None:
    """
    Generates embeddings for a list of text chunks.
    Returns a list of embeddings (each embedding is a list of floats).
    Returns an empty list if an error occurs.
    """
    try:
        response = ollama_client.embeddings.create(
            input=chunks,
            model=ollama_embedding_model,
        )
        return [d.embedding for d in response.data]
    except Exception as e:
        print(f"Error generating embeddings: {e}")
        return []



In [4]:
def readPdfFile(file_path: Path ):
    book1 = PdfReader(bookPath.joinpath(file_path))

    book_text = ""
    count = 0
    for page in book1.pages:
        book_text += page.extract_text()
        count+=1

    print(f"=====>>>\nBook {file_path}, Total Pages : {count}", end="\n====>\n\n")
    return book_text

In [5]:
def chunk_file(book_text, chunk_size=400, overlap=100):
    chunks = []

    start = 0

    while start < len(book_text):
        end = start + chunk_size
        chunks.append(book_text[start:end])

        start += chunk_size - overlap

    return chunks

In [6]:
collection_path = chroma_db_path.joinpath("books")

In [7]:

chroma_client = chromadb.PersistentClient(path=chroma_db_path.joinpath("books"))
try:
    chroma_client.delete_collection("books_collection")
    print("Existing collection 'books_collection' deleted.")
except Exception as e:
    print("Collection 'books_collection' did not exist or could not be deleted.")
collection = chroma_client.create_collection("books_collection")
print("New collection 'books_collection' created.")


Existing collection 'books_collection' deleted.
New collection 'books_collection' created.


In [8]:
def read_and_chunk(filename):
    file_content = readPdfFile(filename)
    chunks = chunk_file(file_content)
    return chunks

book_list = ["AI for Data Science_ Artificial Intelligence Frameworks and Functionality for Deep Learning, Optimization, and Beyond ( PDFDrive ).pdf", "Generative-AI-and-LLMs-for-Dummies.pdf", "MATLAB Deep Learning_ With Machine Learning, Neural Networks and Artificial Intelligence ( PDFDrive ).pdf"]

count = 0
for book in book_list:
    print(f"=== Processing :: {book} ===")
    book_id = f"books_{book}_id"
    chunks = read_and_chunk(book)
    for chunk in chunks:
        embeddings_res = (embed_text_ollama_func(chunk))
        if embeddings_res != []:
            collection.add(
                ids = [book_id],
                metadatas = [{"book_id": book_id}],
                embeddings = embeddings_res,
                documents = [chunk],
            )


=== Processing :: AI for Data Science_ Artificial Intelligence Frameworks and Functionality for Deep Learning, Optimization, and Beyond ( PDFDrive ).pdf ===
=====>>>
Book AI for Data Science_ Artificial Intelligence Frameworks and Functionality for Deep Learning, Optimization, and Beyond ( PDFDrive ).pdf, Total Pages : 231
====>

=== Processing :: Generative-AI-and-LLMs-for-Dummies.pdf ===
=====>>>
Book Generative-AI-and-LLMs-for-Dummies.pdf, Total Pages : 52
====>

=== Processing :: MATLAB Deep Learning_ With Machine Learning, Neural Networks and Artificial Intelligence ( PDFDrive ).pdf ===
=====>>>
Book MATLAB Deep Learning_ With Machine Learning, Neural Networks and Artificial Intelligence ( PDFDrive ).pdf, Total Pages : 162
====>



In [9]:
question = "explain Comparison of Cost Functions?"

In [10]:
response  = collection.query(
    query_embeddings= embed_text_ollama_func(question), # embed_text_func now returns a list, so take the first element
    n_results= 10,
    # where_document={"$contains": f"{question}"},
)
print(len(response))
print(dump_as_json(response))
# for doc in response["documents"][0]:
#     print(doc)

8
{
    "ids": [
        [
            "books_MATLAB Deep Learning_ With Machine Learning, Neural Networks and Artificial Intelligence ( PDFDrive ).pdf_id",
            "books_AI for Data Science_ Artificial Intelligence Frameworks and Functionality for Deep Learning, Optimization, and Beyond ( PDFDrive ).pdf_id",
            "books_Generative-AI-and-LLMs-for-Dummies.pdf_id"
        ]
    ],
    "embeddings": null,
    "documents": [
        [
            "MATLAB  \nDeep Learning\nWith Machine Learning, Neural  \nNetworks and Artificial Intelligence\n\u2014\nPhil KimMATLAB Deep \nLearning\nWith Machine Learning, Neural \nNetworks and Artificial Intelligence\nPhil KimMATLAB Deep Learning: With Machine Learning, Neural Networks and Artificial Intelligence\nPhil Kim     \nSeoul, Soul-t'ukpyolsi, Korea (Republic of ) \nISBN-13 (pbk): 978-1-4842-2844-9  ISBN-13 (electron",
            "Contents\n1.\t\nIntroduction\n1.\t\nAbout\tAI\n2.\t\nAI\tfacilitates\tdata\tscience\n3.\t\nAbout\tthe\tboo